# Consolidated Preprocess Code

### Step1: Clean Code 
### Step2: Fixed Effect Feasibility For Tranche 


# Step1: Cleaning Steps Overview


### 1. Import Data
- Load the raw Dealscan dataset into a pandas DataFrame.

---

### 2. Currency Filter (Ignored for now)
- **Keep only U.S. Dollar-denominated loans.**
- Use the **`Deal_Currency`** column and retain only rows where the value is **`'US Dollar'`** or **`'U.S. Dollar'`**.
- **DataFrame name after this step:** `df_usd`

---

### 3. Bank Nationality Filter (Ignored for now)
- **Filter out non-U.S. banks** using the **`Lender_Parent_Operating_Country`** column.
- Keep only rows where the value is **`'United States'`**.
- **DataFrame name after this step:** `df_usd_usbanks`

---

### 4. Borrower Industry Dummies
- **Create two dummy variables based on `Major_industry_group`:**
  - **`is_financial_borrower = 1`** if the borrower is in **`'Financial Services'`**
  - **`is_utility_borrower = 1`** if the borrower is in **`'Utilities'`**

---

### 5. Deal Status Filter
- **Exclude deals where `Phase` equals `'In Process'`** 
- **DataFrame name after this step:** `df_usd_usbanks_noProgress`

---

### 6. Zero or Negative Loan Amounts
- **Exclude loans where `Deal_Amount` is less than or equal to 0.**
- **DataFrame name after this step:** `df_usd_usbanks_noProgress_positiveAmount`

---

### 7. Deal Purpose Dummy
- **Create the following dummy variable from `Deal_Purpose`:**
  - **`Deal_purpose_LBO_buyout = 1`** if the purpose is:
    - `'Leveraged Buyout'`
    - `'Sponsored Buyout'`
    - `'Takeover'`
    - `'Acquisition'`
    - `'Merger'`

---

## Final Output
- The cleaned dataset will be saved as: **`2021jan_2024sept_cleaned.csv`**

## STEP 1: Load Data


In [ ]:
import pandas as pd
df = pd.read_csv("2021jan_2024sept.csv")
initial_records = len(df)
print(f"Initial records: {initial_records}")

## STEP 1.5: Explore whether U.S. parent banks participate in non-USD loans


In [ ]:
# First, clean the currency column for consistency
df['Deal_Currency_clean'] = df['Deal_Currency'].str.strip().str.lower()

# Filter rows where the parent bank is based in the United States
df_us_parents = df[df['Lender_Parent_Operating_Country'] == 'United States']

# Summary: total deals by U.S. parent banks
total_us_deals = len(df_us_parents)

# Count how many of these deals are non-USD
usd_labels_clean = ['us dollar', 'u.s. dollar']
us_deals_non_usd = df_us_parents[~df_us_parents['Deal_Currency_clean'].isin(usd_labels_clean)]
num_non_usd_us_banks = len(us_deals_non_usd)

# Calculate percentage
pct_non_usd_us_banks = (num_non_usd_us_banks / total_us_deals) * 100

# Print summary
print(f"Total deals by U.S. parent banks: {total_us_deals}")
print(f"Non-USD deals by U.S. parent banks: {num_non_usd_us_banks}")
print(f"Percentage of U.S. bank deals that are non-USD: {pct_non_usd_us_banks:.2f}%")

# Show top currencies used by U.S. banks
print("\nTop currencies used by U.S. parent banks:")
print(df_us_parents['Deal_Currency_clean'].value_counts())

## STEP 2: Filter to only USD-denominated loans (Ignored For Now)
### keep loans where Deal_Currency is 'US Dollar' or 'U.S. Dollar'

check unique values of currency

In [ ]:
# print("Unique values in Deal_Currency column:")
# print(df['Deal_Currency'].unique())

In [ ]:
# # STEP 1 (Revised): Clean currency column and re-filter for USD loans

# # Standardize the Deal_Currency values: strip spaces and convert to lowercase
# df['Deal_Currency_clean'] = df['Deal_Currency'].str.strip().str.lower()

# # Define acceptable cleaned USD labels
# usd_labels_clean = ['us dollar', 'u.s. dollar']

# # Apply the cleaned filter
# df_usd = df[df['Deal_Currency_clean'].isin(usd_labels_clean)]

# # Recalculate counts
# pre_usd_count = len(df)
# post_usd_count = len(df_usd)
# num_non_usd_eliminated = pre_usd_count - post_usd_count
# pct_non_usd_eliminated = (num_non_usd_eliminated / pre_usd_count) * 100

# print(f"Before USD filter (cleaned): {pre_usd_count} records")
# print(f"After USD filter (cleaned): {post_usd_count} records retained")
# print(f"Non-USD loans eliminated: {num_non_usd_eliminated} ({pct_non_usd_eliminated:.2f}%)")

### Verify Deal_Currency

In [ ]:
# print(df_usd['Deal_Currency'].unique())

## STEP 3: Filter to only U.S.-parented banks (Ignored for Now)

check unique values of Lender_parent_operating_country 

In [ ]:
# print("Unique values in Lender_Parent_Operating_Country column:")
# print(df['Lender_Parent_Operating_Country'].unique())

In [ ]:
# # Keep only rows where Lender_Parent_Operating_Country is exactly 'United States'
# pre_bank = len(df)
# df_usd_usbanks = df[df['Lender_Parent_Operating_Country'] == 'United States']
# post_bank = len(df_usd_usbanks)

# # Print filter summary
# print(f"Before filtering for U.S. parent banks: {pre_bank} records")
# print(f"After filtering for U.S. parent banks: {post_bank} records retained")
# print(f"Non-U.S. bank records eliminated: {pre_bank - post_bank} ({(pre_bank - post_bank) / pre_bank:.2%})")

### Verify Lender_Parent_Operating_Country

In [ ]:
print(df['Lender_Parent_Operating_Country'].unique())

## STEP 4: Create Borrower Industry Dummies

check Major_industry_group unique values

In [ ]:
df['Major_Industry_Group'].unique()

In [ ]:
# Create borrower industry dummies
# is_financial_borrower = 1 if 'Financial Services'
# is_utility_borrower = 1 if 'Utilities'

df.loc[:, 'is_financial_borrower'] = (
    df['Major_Industry_Group'] == 'Financial Services'
).astype(int)

df.loc[:, 'is_utility_borrower'] = (
    df['Major_Industry_Group'] == 'Utilities'
).astype(int)


### Verify Dummies

In [ ]:
df[['is_financial_borrower', 'is_utility_borrower', 'Major_Industry_Group']]

## STEP 5: Exclude in-progress deals

check Phase variable unique values

In [ ]:
df['Phase'].unique()

In [ ]:
pre_phase = len(df)

# Drop rows where 'Phase' is 'In Process' (case-insensitive match)
df_usd_usbanks_noProgress = df[df['Phase'].str.lower() != 'in process']

post_phase = len(df_usd_usbanks_noProgress)

# Print summary
print(f"Before filtering in-process deals: {pre_phase} records")
print(f"After excluding in-process deals: {post_phase} records retained")
print(f"In-process deals excluded: {pre_phase - post_phase} ({(pre_phase - post_phase) / pre_phase:.2%})")

In [ ]:
df_usd_usbanks_noProgress['Phase'].unique()

## STEP 6: Exclude loans with zero or negative Deal_Amount


In [ ]:
pre_amount = len(df_usd_usbanks_noProgress)

# Keep only rows where Deal_Amount is greater than 0
df_usd_usbanks_noProgress_positiveAmount = df_usd_usbanks_noProgress[df_usd_usbanks_noProgress['Deal_Amount'] > 0]

post_amount = len(df_usd_usbanks_noProgress_positiveAmount)

# Print summary
print(f"Before filtering non-positive loan amounts: {pre_amount} records")
print(f"After excluding zero or negative Deal_Amount: {post_amount} records retained")
print(f"Records eliminated: {pre_amount - post_amount} ({(pre_amount - post_amount) / pre_amount:.2%})")

### Verify Deal Amount has positive values

In [ ]:
df_usd_usbanks_noProgress_positiveAmount[df_usd_usbanks_noProgress_positiveAmount['Deal_Amount'] < 0]

## STEP 7: Create dummy for LBO/buyout loan purposes

check Deal_Purpose variable unique values

In [ ]:
df_usd_usbanks_noProgress_positiveAmount['Deal_Purpose'].unique()

In [ ]:
lbo_keywords = ['Leveraged Buyout', 'Sponsored Buyout', 'Takeover', 'Acquisition', 'Merger']

df_usd_usbanks_noProgress_positiveAmount.loc[:, 'Deal_purpose_LBO_buyout'] = (
    df_usd_usbanks_noProgress_positiveAmount['Deal_Purpose']
    .isin(lbo_keywords)
    .astype(int)
)

# Count and percentage
num_lbo = df_usd_usbanks_noProgress_positiveAmount['Deal_purpose_LBO_buyout'].sum()
total_deals = len(df_usd_usbanks_noProgress_positiveAmount)
percentage_lbo = (num_lbo / total_deals) * 100

print(f"Number of deals flagged as LBO or buyout: {num_lbo} ({percentage_lbo:.2f}%)")

Verify Deal Purpose Filtering

In [ ]:
df_usd_usbanks_noProgress_positiveAmount[['Deal_Purpose',"Deal_purpose_LBO_buyout"]]

## Final Output

In [ ]:
# 📊 Summary of Final Cleaned DataFrame

# Total initial records before any filtering
print(f"Initial number of records before cleaning: {initial_records}")

# Final record count
final_records = df_usd_usbanks_noProgress_positiveAmount.shape[0]
removed_records = initial_records - final_records
removed_percent = (removed_records / initial_records) * 100

# Print dataset shape summary
print(f"✅ Total records in cleaned dataset: {final_records}")
print(f"📐 Total columns: {df_usd_usbanks_noProgress_positiveAmount.shape[1]}")
print(f"❌ Records removed during cleaning: {removed_records} ({removed_percent:.2f}%)")
print("\n")

# ✨ Highlight: Columns added or changed during cleaning
highlight_columns = [
    'is_financial_borrower',         # Created from Major_industry_group
    'is_utility_borrower',           # Created from Major_industry_group
    'Deal_purpose_LBO_buyout'        # Created from Deal_Purpose
]

print("\n✨ Highlight: Custom or modified columns in this dataset:")
for col in highlight_columns:
    if col in df_usd_usbanks_noProgress_positiveAmount.columns:
        print(f" - {col}:")
        print(f"   Unique values: {df_usd_usbanks_noProgress_positiveAmount[col].unique()}")
        print(f"   Value counts:\n{df_usd_usbanks_noProgress_positiveAmount[col].value_counts()}\n")
    else:
        print(f" - {col}: ❌ Not found in DataFrame")

### Make sure Step2 filtering currency and Step3 filtering US banks are ignored

In [ ]:
print(df_usd_usbanks_noProgress_positiveAmount['Lender_Parent_Operating_Country'].unique())

In [ ]:
print(df_usd_usbanks_noProgress_positiveAmount['Deal_Currency'].unique())

In [ ]:
df_usd_usbanks_noProgress_positiveAmount

In [ ]:
df_usd_usbanks_noProgress_positiveAmount.to_csv("2021jan_2024sept_cleaned.csv", index=False)

# Step2: Fixed Effect Feasibility For Tranche 


## Fixed Effects Feasibility Analysis of Syndicated Loan Data

This step performs a detailed analysis of syndicated loan data (Dealscan) to assess whether tranche level of analysis has sufficient time variation and lender heterogeneity to conduct fixed-effects analysis. 

---

## Objectives

1. **Measure time variation and lender heterogeneity:**
   - For tranches (loan components).
2. **Check fixed-effects feasibility for tranche level analysis.**
4. **Visualize participation over time for comparison.**

---

## Workflow Overview

✅ **1. Load and Parse Data**  
- Load the cleaned Dealscan dataset.
- Parse `Tranche_Active_Date` as a datetime field.

✅ **2. Create Firm-Year Variable**  
- Combine Borrower ID and Year into a "Firm-Year" identifier (e.g., `12345_2020`).
- This measures whether multiple banks participated in lending to the same borrower in the same year.

✅ **3. Measuring Key Metrics**
For each unit (tranche), calculate:
- **% with multiple banks:** proportion of units involving >1 lender.
- **% with multiple active dates:** proportion of units amended or reactivated over time.
- **% meeting both criteria:** units with both lender heterogeneity and time variation.
- **% of firm-years with multiple banks:** borrower-years with more than one bank involved.

These metrics assess feasibility of fixed-effects estimation.

✅ **4. Tranche-Level Analysis**
- Treat each unique `LPC_Tranche_ID` as a unit.
- Compute metrics described above.
- **Filter tranches** that:
  - Have ≥2 unique active dates.
  - Involve >1 bank.
  - Include at least 1 U.S. bank.
- Summarize retained tranches and visualize per quarter.
---

## Outputs

- **Summary tables** showing counts of retained tranches or deals, unique borrowers, and unique U.S. banks.
- **Feasibility metrics** for fixed-effects regressions.
- **Plots per quarter** with annotated counts of U.S. banks and units.

Results of comparing Tranche_Active_Date VS Deal_Input_Date
Total deals analyzed: 28568

Counts:
- Deals where Tranche_Active_Date count > Deal_Input_Date count: 441
- Deals where counts are equal: 28127
- Deals where Tranche_Active_Date count < Deal_Input_Date count: 0

Percentages:
- % Tranche > Input: 1.54%
- % Equal counts: 98.46%
- % Tranche < Input: 0.00%

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =============================
# CONFIGURATION
# =============================

# File containing cleaned Dealscan data
DATA_FILE = "2021jan_2024sept_cleaned.csv"

# Column names in the dataset
ID_COLUMNS = {
    "tranche": "LPC_Tranche_ID",   # Unique identifier for each tranche
    "deal": "LPC_Deal_ID"          # Unique identifier for each deal
}
DATE_COLUMN = "Tranche_Active_Date"          # Activation or amendment date of each tranche
INPUT_DATE_COLUMN = "Deal_Input_Date"        # NEW: for second analysis
BANK_COLUMN = "Lender_Parent_Id"             # ID of the parent bank
US_BANK_COLUMN = "Lender_Parent_Operating_Country"  # Country of the parent bank
BORROWER_COLUMN = "Borrower_Id"              # Borrower company ID

# =============================
# HELPER FUNCTIONS
# =============================

def load_data(filepath):
    """
    Load Dealscan data from CSV.
    Automatically parses Tranche_Active_Date and Deal_Input_Date as datetime columns.
    """
    df = pd.read_csv(filepath, parse_dates=[DATE_COLUMN, INPUT_DATE_COLUMN])
    return df

def create_firm_year(df, date_col):
    """
    Create Firm-Year variable for grouping.
    ---------------------------------------------------
    This combines:
        - Borrower_Id (the firm)
        - Year extracted from the date column
    Into a single string like "12345_2020".
    This allows us to count:
        - How many different banks lent to the same borrower
          in the same year.
    """
    df["Year"] = df[date_col].dt.year
    df["Firm_Year"] = df[BORROWER_COLUMN].astype(str) + "_" + df["Year"].astype(str)
    return df

def summarize_fixed_effects(df, unit_col, date_col):
    """
    Compute feasibility metrics to assess whether fixed effects regressions
    are possible with this dataset.
    ---------------------------------------------------
    For each tranche, calculates:
        1. % with >1 unique bank involved
        2. % with >1 unique active date
        3. % meeting both conditions (multiple banks & dates)
    Separately, computes:
        4. % of Firm-Year pairs with multiple banks
    These statistics help evaluate:
        - Whether there is sufficient variation within units and over time
          to identify causal effects of deposits on lending.
    """
    # Count # of unique banks per unit
    bank_counts = df.groupby(unit_col)[BANK_COLUMN].nunique()
    pct_multiple_banks = (bank_counts > 1).mean() * 100

    # Count # of unique dates per unit
    date_counts = df.groupby(unit_col)[date_col].nunique()
    pct_multiple_dates = (date_counts > 1).mean() * 100

    # Units with both multiple banks and multiple dates
    combined = (bank_counts > 1) & (date_counts > 1)
    pct_combined = combined.mean() * 100

    # Firm-Year pairs with multiple banks
    firm_year_counts = df.groupby("Firm_Year")[BANK_COLUMN].nunique()
    pct_firm_year_multiple_banks = (firm_year_counts > 1).mean() * 100

    # Print summary
    print(f"\n🔍 Fixed Effects Feasibility Check ({unit_col}) using {date_col}:")
    print(f"1. Units with multiple banks               : {pct_multiple_banks:.2f}%")
    print(f"2. Units with multiple active dates        : {pct_multiple_dates:.2f}%")
    print(f"3. Units with both multiple banks & dates  : {pct_combined:.2f}%")
    print(f"4. Firm-year pairs with multiple banks     : {pct_firm_year_multiple_banks:.2f}%")

    return bank_counts, date_counts

def plot_counts_per_quarter(df, unit_col, date_col, title_prefix):
    """
    Generate a dual-axis plot showing:
    ---------------------------------------------------
    - Number of unique U.S. banks per quarter (bar chart)
    - Number of unique units (tranches or deals) per quarter (line chart)
    Each bar and point is labeled with the count for clarity.
    """
    # Create Quarter variable for grouping
    df["Quarter"] = df[date_col].dt.to_period("Q").astype(str)

    # Count U.S. banks per quarter
    us_banks_q = (
        df[df[US_BANK_COLUMN] == "United States"]
        .groupby("Quarter")[BANK_COLUMN]
        .nunique()
    )

    # Count units of tranches per quarter
    units_q = df.groupby("Quarter")[unit_col].nunique()

    # Create the figure
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Bar plot for U.S. banks
    bars = ax1.bar(us_banks_q.index, us_banks_q.values, alpha=0.6, label="U.S. Banks")
    ax1.set_ylabel("Number of U.S. Banks")
    ax1.set_title(f"{title_prefix}: U.S. Banks and {unit_col} per Quarter")
    ax1.tick_params(axis="x", rotation=45)

    # Annotate each bar with count
    for bar in bars:
        height = bar.get_height()
        ax1.annotate(
            f"{int(height)}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center", va="bottom", fontsize=8
        )

    # Line plot for unit counts
    ax2 = ax1.twinx()
    line = ax2.plot(units_q.index, units_q.values, color="red", marker="o", label=f"{unit_col} count")
    ax2.set_ylabel(f"Number of {unit_col}")

    # Annotate each point with count
    for x, y in zip(units_q.index, units_q.values):
        ax2.annotate(
            f"{int(y)}",
            xy=(x, y),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center", va="bottom",
            fontsize=8, color="red"
        )

    # Add legend and layout adjustments
    fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.9))
    plt.tight_layout()
    plt.show()

def report_summary(df, unit_col):
    """
    Print a concise summary of the dataset after filtering.
    ---------------------------------------------------
    Includes:
        - Number of unique units (tranches or deals)
        - Number of unique borrowers
        - Number of unique U.S. banks
    """
    n_units = df[unit_col].nunique()
    n_borrowers = df[BORROWER_COLUMN].nunique()
    n_us_banks = df[df[US_BANK_COLUMN] == "United States"][BANK_COLUMN].nunique()

    print("\n✅ Summary:")
    print(f"• Unique {unit_col}: {n_units}")
    print(f"• Unique borrowers: {n_borrowers}")
    print(f"• U.S. banks: {n_us_banks}")

# =============================
# MAIN WORKFLOW
# =============================


In [ ]:
if __name__ == "__main__":
    # Load the full dataset
    df_all = load_data(DATA_FILE)

    # ============================================
    # ========== TRANCHE-LEVEL ANALYSIS ==========
    # ============================================
    print("\n================ Tranche-Level Analysis ================")

    df_tranche = create_firm_year(df_all.copy(), DATE_COLUMN)

    # Compute fixed effects feasibility
    bank_counts_t, date_counts_t = summarize_fixed_effects(
        df_tranche,
        ID_COLUMNS["tranche"],
        DATE_COLUMN
    )

    # Tranche-level summary
    df_tranche_filtered = df_tranche.groupby(ID_COLUMNS["tranche"]).filter(
        lambda x: x[DATE_COLUMN].nunique() >= 2 and
                  x[BANK_COLUMN].nunique() > 1 and
                  "United States" in x[US_BANK_COLUMN].values
    )
    report_summary(df_tranche_filtered, ID_COLUMNS["tranche"])


    # ================ PLOTTING ==================
    plot_counts_per_quarter(df_tranche_filtered, ID_COLUMNS["tranche"], DATE_COLUMN, "Tranche-Level")